In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT
if not (PROJECT_ROOT / "HFTNET").exists():
    for parent in [CURRENT, *CURRENT.parents]:
        if (parent / "HFTNET").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("Projeto:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())


In [ ]:
from HFTNET import (
    build_folds_csv,
    build_inbreast_csv,
    make_run_dirs,
    parse_breakhis_dataset,
    run_breakhis_baseline_holdout,
    run_transfer_breakhis_to_inbreast,
)

BREAKHIS_BASE_PATH = PROJECT_ROOT / "BrestCancer Datasets" / "BreaKHis" / "BreaKHis_v1" / "BreaKHis_v1" / "histology_slides" / "breast"
BREAKHIS_CSV = PROJECT_ROOT / "HFTNET" / "breakhis_multiclass_folds.csv"
INBREAST_CSV_SOURCE = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "INbreast.csv"
INBREAST_DICOM_DIR = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "AllDICOMs"
INBREAST_FOLDS_CSV = PROJECT_ROOT / "HFTNET" / "inbreast_multiclass_folds.csv"


In [ ]:
df_breakhis = parse_breakhis_dataset(str(BREAKHIS_BASE_PATH), mode="multiclass")
build_folds_csv(df_breakhis, str(BREAKHIS_CSV), n_splits=5, seed=42)

build_inbreast_csv(
    inbreast_csv_path=str(INBREAST_CSV_SOURCE),
    dicom_dir=str(INBREAST_DICOM_DIR),
    out_csv=str(INBREAST_FOLDS_CSV),
    mode="multiclass",
    n_splits=5,
    seed=42,
)


In [ ]:
source_run_dirs = make_run_dirs(run_prefix="hftnet_breakhis_source")
source_model, source_history, source_outputs, source_run_dirs = run_breakhis_baseline_holdout(
    base_path=str(BREAKHIS_BASE_PATH),
    mode="multiclass",
    val_fraction=0.2,
    epochs=5,
    batch_size=8,
    run_dirs=source_run_dirs,
    img_size=224,
    balance=True,
)

print("Run fonte:", source_run_dirs["out_dir"])


In [ ]:
target_run_dirs = make_run_dirs(run_prefix="hftnet_breakhis_to_inbreast_tl")
target_model, target_history, target_outputs, target_run_dirs = run_transfer_breakhis_to_inbreast(
    inbreast_csv_path=str(INBREAST_FOLDS_CSV),
    val_fraction=0.2,
    epochs=5,
    batch_size=8,
    run_dirs=target_run_dirs,
    img_size=224,
    balance=True,
    freeze_except_classifier=True,
)

print("Run alvo:", target_run_dirs["out_dir"])
